# Compare_Sycamore_Runs

**Compares finished Lower Sycamore Wash tRIBS runs with the USGS gauge (09510200), water year 2017.**

This notebook only **reads** results. It builds nothing, changes no model file, and does not need pytRIBS. It was split off from `Make_Sycamore_Model.ipynb` on 2026-10-22 (the cells that read `Srf` instead of the routed flow were left behind on purpose).

### Before you start
- Run it from `workspaces/Sycamore/lab/`. Results are read from `results/`, and the gauge record from `../init_data/obs/sycamore_09510200_obs_flow_WY2017.csv` (made by `make_obs_flow.py`).
- Each run you compare must have finished in tRIBS: an `.mrf` file and an `_Outlet.qout` file for it in `results/`.

### How to use it
1. Run the **Imports** cell and the **working folder** cell.
2. In the **comparison cell**, add one line to `RUNS` for each new run (the run name, then the date and time it started), and run it.
3. The **daily highest flow** and **rain vs runoff** cells reuse what the comparison cell loaded, so run them after it, in the same session.
4. The **screening** cells read two things that `Make_Sycamore_Model.ipynb` creates once: the SOLUS Ks grid (`../init_data/solus/Ks_0-5cm.asc`) and the basin outline (`data/preprocessing/Sycamore_boundary.shp`). They do not need that notebook to have run in the same session. Run the two "values and helper functions" cells just before them.
5. The **Voronoi polygon check** is separate. It needs only the two polygon files of a finished tRIBS run (see its heading), not the comparison cell.

The comparison cell needs only the Imports cell. After a restarted Codespace or kernel, run Imports, the working-folder check, then the comparison cell.

## Imports

An *import* makes a Python library available to the notebook. **All imports for this notebook are in the next cell**, so run it first in every session (after a kernel restart too).

In [ ]:
# Standard library
import os
import glob

# Numbers, tables and maps
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import Polygon

# Figures
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Check the working folder

This notebook finds `results/` and `../init_data/` relative to the folder it runs in, so it has to run from `workspaces/Sycamore/lab/`. The next cell only reports; it does not stop the notebook.

In [ ]:
if os.path.basename(os.getcwd()) == 'lab':
    print('Working folder OK:', os.getcwd())
else:
    print(f"WARNING: the working folder is '{os.getcwd()}'. This notebook expects to run from workspaces/Sycamore/lab/.")

## Compare runs with the gauge (routed outlet flow)

This is the main comparison cell. For every run listed in its `RUNS` block it reads
- the **routed outlet flow** from `results/{RESULTS_NAME}_Outlet.qout`: the flow after it has travelled down the channels (and lost water, if channel loss was on). This is the series to compare with the gauge;
- the hourly basin averages from the `.mrf` file (rain, water-table depth, snow, and `Srf`, the runoff made on the hillslopes **before** routing, which is shown for reference only).

It compares them with the USGS gauge record and prints a January storm-week table (peak, timing and volume, with their errors), a month-by-month volume table for the runs that start on Oct 1, and plots of rain, flow and water-table depth.

- **To add a run, add one line to `RUNS`:** the run name, then the date and time it started (Arizona time). Then rerun the cell. A run whose files are missing stops the cell with a message that names the file.
- The cell also creates the names `runs`, `qouts`, `q_obs`, `volume_pieces` and `volume`, which the next two cells reuse, so run them after it in the same session.

It replaces the older comparison cells, which plotted `Srf` instead of the routed flow and so could not show the effect of channel loss.

In [ ]:
# ---- Compare Sycamore runs with the gauge, using the ROUTED OUTLET FLOW (version 2 of the compare cell) ----------
# What changed from compare_spinup_cell.py:
#   * The model flow is now Qstrm from results/{RESULTS_NAME}_Outlet.qout (the flow that has been routed down the
#     channels, with channel loss applied). The old cell used Srf_m3_s from the .mrf, which is runoff generated
#     BEFORE routing: channel loss cannot change it, and it peaks hours before the gauge.
#   * Srf is still shown, as a thin dashed line (and in the table, marked "unrouted"), for reference only.
# Read-only: this cell writes nothing.

# Each run: its RESULTS_NAME, and the date and time it started from (Arizona time).
RUNS = {
#    'Sycamore_jan':                        '2017-01-18 00:00',   # snow off, original bedrock, water table 0.95 of bedrock
#    'Sycamore_jan_snow':                   '2017-01-18 00:00',   # snow on
#    'Sycamore_jan_snow_bed1':              '2017-01-18 00:00',   # bedrock +1 m (water table file was stale)
#    'Sycamore_jan_snow_bed1gw':            '2017-01-18 00:00',   # bedrock +1 m, water table recomputed
#    'Sycamore_octjan_snow_bed1gw':         '2016-10-01 00:00',   # started Oct 1 (spin-up)
#    'Sycamore_jan_ks0p3':                  '2017-01-18 00:00',   # Ks x 0.3, water table 0.82 of bedrock, channel loss ON (70 mm/hr)
#    'Sycamore_jan_ks0p3_nocl':             '2017-01-18 00:00',   # same, channel loss OFF
#    'Sycamore_jan_ks0p2_nocl':             '2017-01-18 00:00',   # Ks x 0.2, channel loss OFF
    'Sycamore_jan_ks0p2_cv6_nocl':          '2017-01-18 00:00',   # Ks x 0.2, cv (KINEMVELCOEFF) 6, channel loss OFF
    'Sycamore_octjan_ks0p2_cv6_nocl':       '2016-10-01 00:00',   # spin-up Oct 1 to Jan 25, Ks x 0.2, cv 6, channel loss OFF
    'Sycamore_octmar_ks0p2_cv6_nocl':       '2016-10-01 00:00',   # Oct 1 to Apr 1, loss OFF, Ks x 0.2, cv 6
    'Sycamore_octmar_ks0p2_cv6_kchan2':     '2016-10-01 00:00',
    'Sycamore_octmar_ks0p2_cv6_kchan5':     '2016-10-01 00:00',
    'Sycamore_octmar_ks0p18_cv6_kchan5':    '2016-10-01 00:00',
    'Sycamore_octmar_ks0p2_cv6_psib0p5_nocl': '2016-10-01 00:00',
    'Sycamore_octmar_ks0p2_cv6_psib0p5_f0p5_nocl': '2016-10-01 00:00',
}
WINDOW_START, WINDOW_END = '2017-01-18 00:00', '2017-01-25 00:00'    # the January storm week
SECOND_STORM_FROM = '2017-01-23 12:00'     # the table also reports the biggest flow after this time
OBS_FILE = '../init_data/obs/sycamore_09510200_obs_flow_WY2017.csv'


def read_qout(path, start):
    """Routed outlet flow (m3/s) as a time series. tRIBS 6.0.0 writes a comma-delimited file with a header line."""
    with open(path) as fh:
        fh.readline()
        second = fh.readline()
    if ',' in second:
        raw = pd.read_csv(path, sep=',', header=0)
    else:
        raw = pd.read_csv(path, sep=r'\s+', skiprows=1, header=None)
    raw = raw.iloc[:, :2].copy()
    raw.columns = ['Time_hr', 'Q']
    raw = raw.apply(pd.to_numeric, errors='coerce').dropna()
    return pd.Series(raw['Q'].to_numpy(), index=pd.Timestamp(start) + pd.to_timedelta(raw['Time_hr'].to_numpy(), unit='h'))


def load_run(name, start):
    mrf_files = sorted(glob.glob(f'**/{name}[0-9]*.mrf', recursive=True))
    if not mrf_files:
        raise FileNotFoundError(f'No .mrf file found for {name!r}. Files found: {glob.glob("**/*.mrf", recursive=True)}')
    sim = pd.read_csv(mrf_files[-1])
    sim = sim[sim['RainMin_mm_hr'] < 9000]                     # drop the unfilled last row
    sim.index = pd.Timestamp(start) + pd.to_timedelta(sim['Time_hr'], unit='h')
    qout_files = glob.glob(f'**/{name}_Outlet.qout', recursive=True)
    if len(qout_files) != 1:
        raise FileNotFoundError(f'Expected exactly one {name}_Outlet.qout, found {qout_files}. '
                                f'All .qout files: {sorted(glob.glob("**/*.qout", recursive=True))}')
    return sim, read_qout(qout_files[0], start)


def volume_pieces(q):
    """Water volume (m3) of each time step: trapezoid rule on a flow series (m3/s) indexed by time.
    Each value is labeled with the time its step STARTS. Steps that touch a blank are skipped."""
    t = q.index.values.astype('datetime64[s]').astype('int64')
    v = q.values.astype(float)
    good = ~np.isnan(v[1:]) & ~np.isnan(v[:-1])
    return pd.Series(np.where(good, 0.5 * (v[1:] + v[:-1]) * np.diff(t), 0.0), index=q.index[:-1])


def volume(q, a=None, b=None):
    """Total volume (m3) from time a up to (not including) time b. Leave a and b out for the whole series."""
    p = volume_pieces(q)
    keep = np.ones(len(p), dtype=bool)
    if a is not None:
        keep &= p.index >= pd.Timestamp(a)
    if b is not None:
        keep &= p.index < pd.Timestamp(b)
    return float(p[keep].sum())


loaded = {name: load_run(name, start) for name, start in RUNS.items()}
runs = {name: sim for name, (sim, q) in loaded.items()}          # the .mrf tables (basin averages, hourly)
qouts = {name: q for name, (sim, q) in loaded.items()}           # the routed outlet flow (m3/s)
for name in runs:
    q = qouts[name]
    print(f'{name}: outlet flow {q.index[0]:%b %d %Y %H:%M} to {q.index[-1]:%b %d %Y %H:%M} ({len(q)} rows); '
          f'.mrf {len(runs[name])} hourly rows')

obs = pd.read_csv(OBS_FILE, parse_dates=['datetime']).set_index('datetime')
q_obs = obs['q_cms'].loc[min(r.index[0] for r in runs.values()):max(r.index[-1] for r in runs.values())]
q_obs_week = q_obs.loc[WINDOW_START:WINDOW_END]

# ---- Table 1: the January storm week, every run vs the gauge ---------------------------------------------
t_gauge = q_obs_week.idxmax()
gauge_peak = q_obs_week.max()
gauge_vol = volume(q_obs, WINDOW_START, WINDOW_END) / 1e6
rows = {'gauge': {'peak (m3/s)': gauge_peak, 'peak vs gauge (h)': 0.0,
                  'volume (M m3)': gauge_vol,
                  'biggest flow after 2nd-storm time (m3/s)': q_obs.loc[SECOND_STORM_FROM:WINDOW_END].max()}}
for name in runs:
    q = qouts[name].loc[WINDOW_START:WINDOW_END]
    week = runs[name].loc[WINDOW_START:WINDOW_END]
    if q.empty or week.empty:
        print(f'(skipping {name} in the January table: the run does not cover {WINDOW_START} to {WINDOW_END})')
        continue
    vol = volume(q, WINDOW_START, WINDOW_END) / 1e6
    rows[name] = {'peak (m3/s)': q.max(),
                  'peak vs gauge (h)': (q.idxmax() - t_gauge).total_seconds() / 3600,
                  'peak error vs gauge (%)': (q.max() / gauge_peak - 1) * 100,
                  'volume (M m3)': vol,
                  'volume error vs gauge (%)': (vol / gauge_vol - 1) * 100,
                  'biggest flow after 2nd-storm time (m3/s)': q.loc[SECOND_STORM_FROM:].max(),
                  'channel loss (M m3)': week['ChannelPerc_m3'].sum() / 1e6,
                  'UNROUTED Srf peak (m3/s)': week['Srf_m3_s'].max(),
                  'UNROUTED Srf volume (M m3)': volume(week['Srf_m3_s'], WINDOW_START, WINDOW_END) / 1e6,
                  'water table on Jan 18 (mm)': week['MDGW_mm'].iloc[0],
                  'water table, deepest in week (mm)': week['MDGW_mm'].max(),
                  'water table, shallowest in week (mm)': week['MDGW_mm'].min(),
                  'snow max (cm)': week['AvSWE_cm'].max()}
print('\nJANUARY STORM WEEK (Jan 18 to Jan 25), model flow = routed outlet discharge from the .qout file')
print(pd.DataFrame(rows).to_string(float_format=lambda x: f'{x:.2f}'))      # one column per run
print('Rows marked UNROUTED are the .mrf Srf column: runoff generated on the hillslopes before routing. Reference only.')
print('water table = mean depth to the water table below the surface (mm); a bigger number is deeper.')

# ---- Table 2: the spin-up months, model vs gauge (only runs that start before Jan 18) --------------------------------
for name, sim in runs.items():
    if sim.index[0] >= pd.Timestamp(WINDOW_START):
        continue
    month_model = volume_pieces(qouts[name]).resample('MS').sum() / 1e6
    month_srf = volume_pieces(sim['Srf_m3_s']).resample('MS').sum() / 1e6
    month_gauge = volume_pieces(q_obs).resample('MS').sum() / 1e6
    month_rain = sim['MAP_mm_hr'].resample('MS').sum()                # hourly mm/hr x 1 hour = mm
    month_table = pd.DataFrame({'rain (mm)': month_rain, 'gauge flow (M m3)': month_gauge,
                                'model outlet flow (M m3)': month_model, 'unrouted Srf (M m3)': month_srf})
    month_table.index = month_table.index.strftime('%b %Y')
    month_table.loc['total'] = month_table.sum()
    month_table['model / gauge'] = month_table['model outlet flow (M m3)'] / month_table['gauge flow (M m3)']
    print(f'\nMONTH BY MONTH, {name} (last hour of the run: {sim.index[-1]:%b %d %H:%M})')
    print(month_table.to_string(float_format=lambda x: f'{x:.2f}'))
    wt = sim['MDGW_mm']
    print(f'water table: starts {wt.iloc[0]:.0f} mm, is {wt.asof(pd.Timestamp(WINDOW_START)):.0f} mm on Jan 18, '
          f'deepest {wt.max():.0f} mm, shallowest {wt.min():.0f} mm.')
print('\n(The gauge total skips any blank stretch, such as the Jan 1 outage, so it can only be too small.)')

# ---- Pictures: rain, flow, water-table depth ----------------------------------------------------------------------
palette = ['tab:orange', 'tab:red', 'tab:green', 'tab:purple', 'tab:brown', 'tab:cyan', 'tab:pink', 'tab:olive', 'tab:gray']
colors = dict(zip(runs, palette))


def draw(start, end, title, date_format, shade=False):
    fig, (ax_r, ax_q, ax_w) = plt.subplots(3, 1, figsize=(11, 9), sharex=True, gridspec_kw={'height_ratios': [1, 3, 2]})
    longest = max(runs.values(), key=len)
    rain = longest.loc[start:end]
    ax_r.bar(rain.index, rain['MAP_mm_hr'], width=-1/24, color='tab:blue', align='edge')   # each bar covers the hour ENDING at its time
    ax_r.invert_yaxis()
    ax_r.set_ylabel('Rain\n(mm/hr)')
    ax_r.set_title(title)
    qo = q_obs.loc[start:end]
    ax_q.plot(qo.index, qo.values, color='black', lw=1.5, label='gauge (15 min)')
    for name, sim in runs.items():
        part = sim.loc[start:end]
        q = qouts[name].loc[start:end]
        if part.empty:
            continue
        ax_q.plot(q.index, q.values, lw=1.6, color=colors[name], label=name)                       # routed outlet flow
        ax_q.plot(part.index, part['Srf_m3_s'], lw=0.8, ls='--', color=colors[name], alpha=0.8)   # unrouted Srf, same color
        ax_w.plot(part.index, part['MDGW_mm'], lw=1.6, color=colors[name], label=name)
        ax_w.plot(part.index[0], part['MDGW_mm'].iloc[0], 'o', color=colors[name])           # dot = where the run started
    ax_q.plot([], [], lw=0.8, ls='--', color='gray', label='thin dashed = Srf, unrouted runoff (same colors)')
    if shade:                                                                                # mark the January week
        for ax in (ax_r, ax_q, ax_w):
            ax.axvspan(pd.Timestamp(WINDOW_START), pd.Timestamp(WINDOW_END), color='gold', alpha=0.25)
    ax_q.set_ylabel('Flow (m3/s)')
    ax_q.legend(fontsize=8)
    ax_q.grid(alpha=0.3)
    ax_w.set_ylabel('Mean depth to\nwater table (mm)')
    ax_w.invert_yaxis()
    ax_w.grid(alpha=0.3)
    ax_w.xaxis.set_major_formatter(mdates.DateFormatter(date_format))
    plt.tight_layout()
    plt.show()


first_start = min(r.index[0] for r in runs.values())
draw(first_start, WINDOW_END, 'Whole spin-up run vs USGS 09510200 gauge (shaded = the January storm week)',
     '%b %d', shade=True)
draw(WINDOW_START, WINDOW_END, 'January storm week: routed outlet flow (solid) and unrouted Srf (dashed) vs USGS 09510200 gauge',
     '%b %d\n%H:%M')

## Daily highest flow, gauge vs model runs

An add-on to the comparison cell: run it after that cell, in the same session. For every day from `DAY_FROM` to `DAY_TO` on which the gauge or any run reaches `MIN_FLOW` (1 m3/s), it lists the day's highest flow for the gauge and for each run. The monthly table hides *when* flow happens; this one shows storms and their recession tails day by day.

In [ ]:
# ---- Daily highest flow, gauge vs model runs, for the days when anything is flowing (add-on to the compare cell) ------
# Run this AFTER the version-2 compare cell, in the same notebook session. It reuses q_obs and qouts from that cell.
# Read-only: it writes nothing and does not touch any model file.
#
# Why this cell exists: the monthly table hides when flow happens. A "March peak" can turn out to be the tail of a
# storm from the last day of February. Here every day gets its own highest flow, so storms and their tails can be seen.

DAY_FROM, DAY_TO = '2017-01-26', '2017-03-31'   # the days to list (the January flood week is already covered elsewhere)
MIN_FLOW = 1.0                                   # m3/s: list a day if the gauge OR any run reaches this

columns = {'gauge (m3/s)': q_obs.loc[DAY_FROM:DAY_TO + ' 23:59'].dropna().resample('D').max()}
for name, q in qouts.items():
    daily = q.loc[DAY_FROM:DAY_TO + ' 23:59'].resample('D').max()
    if daily.notna().any():                      # skip runs that do not cover these days
        columns[name.replace('Sycamore_', '') + ' (m3/s)'] = daily

days = pd.DataFrame(columns)
days = days[(days >= MIN_FLOW).any(axis=1)]
days.index = days.index.strftime('%b %d')
print(f'DAILY HIGHEST FLOW, {DAY_FROM} to {DAY_TO}, only days when the gauge or a run reaches {MIN_FLOW:g} m3/s')
print(days.to_string(float_format=lambda x: f'{x:.2f}', na_rep='-'))

## Rain vs runoff, day by day and storm by storm

An add-on to the comparison cell: run it after that cell, in the same session. It asks whether the **rain the model was given** can explain why the model is too high in some storms and too low in others. It puts the basin-average radar rain next to the gauge flow and the flow of the runs in `REF_RUNS`, then divides outflow volume by rain volume (the *runoff ratio*) for each storm window in `STORMS`.

To change what is shown, edit `REF_RUNS` (a label and a run name, which must already be in the comparison cell's `RUNS`) and `STORMS` (windows with the start included and the end not included).

In [ ]:
# ---- Rain vs runoff, day by day and storm by storm (add-on to the compare cell) ---------------------------------------
# Run this AFTER the version-2 compare cell, in the same notebook session. It reuses runs, qouts, q_obs, volume_pieces()
# and volume() from that cell. Read-only: it writes nothing and does not touch any model file.
#
# Question it answers: the model is too high on Feb 19 and too low on Feb 28 and in January, whatever Ks and Kchan are.
# Is that because the RAIN the model was given does not match what the gauge's response implies for those storms?
# It puts the basin-average radar rain next to the gauge flow and next to the model flow, and divides outflow by rain
# (outflow volume / rain volume: the "runoff ratio"). The rain is the same in every run, so one run's rain is enough.

DAY_FROM, DAY_TO = '2017-01-18', '2017-03-11'     # days for the day-by-day table
MIN_RAIN, MIN_FLOW = 1.0, 1.0                     # list a day if basin rain >= 1 mm OR the gauge reaches 1 m3/s
BASIN_KM2 = 424.6                                 # watershed area from the build (424.6151 km2)

# Which model runs to show next to the gauge (label: run name, as in the compare cell's RUNS block).
REF_RUNS = {'loss OFF': 'Sycamore_octmar_ks0p2_cv6_nocl',
            'PsiB x0.5': 'Sycamore_octmar_ks0p2_cv6_psib0p5_nocl',
            'f x0.5': 'Sycamore_octmar_ks0p2_cv6_psib0p5_f0p5_nocl'}

# Storm windows: start (included) to end (not included). Edit these after reading the day-by-day table.
# The long windows include each storm's recession tail, because the tails hold much of the gauge volume.
# The cut between the two February storms is Feb 27 00:00: the day-by-day table (first run, 10-21) showed 19 mm of rain
# on Feb 27, which belongs to the Feb 28 storm. The "first days" windows leave the tails out.
STORMS = {'Jan flood + tail':      ('2017-01-18 00:00', '2017-02-03 00:00'),
          'Feb 18-19 storm + tail': ('2017-02-18 00:00', '2017-02-27 00:00'),
          'Feb 27-28 storm + tail': ('2017-02-27 00:00', '2017-03-12 00:00'),
          'Feb 18-20 first days':   ('2017-02-18 00:00', '2017-02-21 00:00'),
          'Feb 27-Mar 2 first days': ('2017-02-27 00:00', '2017-03-03 00:00')}

missing = [n for n in REF_RUNS.values() if n not in runs]
if missing:
    raise KeyError(f'These runs are not in the compare cell RUNS block: {missing}. Add them and rerun the compare cell.')
first = next(iter(REF_RUNS.values()))

# Basin-average rain (mm/hr, so each hourly value is mm). The .mrf labels each hour by the time it ENDS, so move the
# label back one hour: then a value labeled 00:00 on Feb 20 (the hour 23:00 to 24:00 on Feb 19) counts in Feb 19.
rain_hr = runs[first]['MAP_mm_hr'].copy()
rain_hr.index = rain_hr.index - pd.Timedelta(hours=1)

# ---- Table 1: day by day ----------------------------------------------------------------------------------------------
a, b = DAY_FROM, DAY_TO + ' 23:59'
gauge_pieces = volume_pieces(q_obs)
cols = {'rain (mm)': rain_hr.loc[a:b].resample('D').sum(),
        'peak rain (mm/hr)': rain_hr.loc[a:b].resample('D').max(),
        'gauge max (m3/s)': q_obs.loc[a:b].dropna().resample('D').max(),
        'gauge (M m3)': gauge_pieces.loc[a:b].resample('D').sum() / 1e6}
for label, name in REF_RUNS.items():
    cols[f'{label} (M m3)'] = volume_pieces(qouts[name]).loc[a:b].resample('D').sum() / 1e6
cols['unrouted Srf (M m3)'] = volume_pieces(runs[first]['Srf_m3_s']).loc[a:b].resample('D').sum() / 1e6
day_table = pd.DataFrame(cols)
day_table = day_table[(day_table['rain (mm)'] >= MIN_RAIN) | (day_table['gauge max (m3/s)'] >= MIN_FLOW)]
day_table.index = day_table.index.strftime('%b %d')
print(f'DAY BY DAY, {DAY_FROM} to {DAY_TO}: days with basin-average rain of {MIN_RAIN:g} mm or more, or gauge flow of '
      f'{MIN_FLOW:g} m3/s or more')
print(f'(rain = basin average of the radar rain the model was given; "unrouted Srf" is from {first})')
print(day_table.to_string(float_format=lambda x: f'{x:.2f}', na_rep='-'))

# ---- Table 2: storm windows ---------------------------------------------------------------------------------------------
storm_cols = {}
peak_lines = []
for storm, (s0, s1) in STORMS.items():
    r = rain_hr[(rain_hr.index >= pd.Timestamp(s0)) & (rain_hr.index < pd.Timestamp(s1))]
    rain_mm = r.sum()
    rain_vol = rain_mm * BASIN_KM2 / 1000.0              # mm over BASIN_KM2 km2, in million m3
    col = {'rain (mm, basin average)': rain_mm,
           'rain volume (M m3)': rain_vol,
           'gauge outflow (M m3)': volume(q_obs, s0, s1) / 1e6}
    col['gauge / rain'] = col['gauge outflow (M m3)'] / rain_vol if rain_vol > 0 else float('nan')
    for label, name in REF_RUNS.items():
        v = volume(qouts[name], s0, s1) / 1e6
        col[f'{label} outflow (M m3)'] = v
        col[f'{label} / rain'] = v / rain_vol if rain_vol > 0 else float('nan')
    srf = volume(runs[first]['Srf_m3_s'], s0, s1) / 1e6
    col['unrouted Srf (M m3)'] = srf
    col['unrouted Srf / rain'] = srf / rain_vol if rain_vol > 0 else float('nan')
    storm_cols[storm] = col
    if len(r) and r.max() > 0:
        peak_lines.append(f'  {storm}: heaviest hour {r.max():.2f} mm/hr (basin average), hour ending '
                          f'{r.idxmax() + pd.Timedelta(hours=1):%b %d %H:%M}')
print('\nSTORM WINDOWS (start included, end not included):')
for storm, (s0, s1) in STORMS.items():
    print(f'  {storm}: {s0} to {s1}')
print(pd.DataFrame(storm_cols).to_string(float_format=lambda x: f'{x:.3f}', na_rep='-'))
print('Heaviest hour in each window:')
print('\n'.join(peak_lines))
print('\n"/ rain" = outflow volume divided by rain volume. Rain is a basin average, so a patchy storm is smoothed out.')
print('The gauge total skips blank stretches, so a gauge volume can only be too small.')

## Screening: can the radar rain alone explain the gauge volumes?

The two screening cells that follow are quick checks that need **no model run**. They test the infiltration idea: where rain falls faster than the soil can take it up (the rain rate is above Ks), the extra becomes runoff.

They read files that the build notebook creates once: the SOLUS Ks grid (written by its soil workflow into `../init_data/solus/`) and the basin outline (saved by its delineation step). Run `Make_Sycamore_Model.ipynb` at least once first; it does not have to have run in the same session. The next two cells set up the names the screening cells use.

### Values and helper functions the screening cells need
The first cell below repeats a few values from the Settings cell of `Make_Sycamore_Model.ipynb` (**if you change them there, change them here**), finds the radar folder the same way the build notebook does, and reads the basin outline that the build notebook's delineation step saved (`data/preprocessing/Sycamore_boundary.shp`). The second cell defines the two small helper functions `values_in_basin` and `valid_basin_values`, the same ones the build notebook uses.

In [ ]:
# ---- Values repeated from Settings in Make_Sycamore_Model.ipynb. If you change them there, change them here. ----
NAME = 'Sycamore'
SOLUS_DIR = '../init_data'                # SOLUS grids are in {SOLUS_DIR}/solus
RADAR_ROOT = '~/mrms_wk_wy2017'           # searched for the folder holding the END-labeled radar files
RADAR_DIR = None                          # None = find it under RADAR_ROOT. Or give the folder explicitly.
RADAR_BASENAME = 'radar_'
RADAR_EXT = 'asc'
RADAR_LAST_FILE = 'radar_1001201700.asc'  # only the end-labeled set has this one (hour ending 10/01/2017 00:00)

# ---- The radar folder: the one under RADAR_ROOT that contains RADAR_LAST_FILE (same search as the build notebook) ----
if RADAR_DIR is None:
    root = os.path.expanduser(RADAR_ROOT)
    hits = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, '**', RADAR_LAST_FILE), recursive=True)})
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected exactly one folder under {root} that contains {RADAR_LAST_FILE}, found {hits}. '
                                'Set RADAR_DIR to the folder in this cell.')
    RADAR_DIR = os.path.abspath(hits[0])
else:
    RADAR_DIR = os.path.abspath(os.path.expanduser(RADAR_DIR))
print('Radar folder:', RADAR_DIR)

# ---- The basin outline saved by the build notebook's delineation step (the file keeps all its pieces, as there) ----
path_to_watershed = f'data/preprocessing/{NAME}_boundary.shp'
basin_shapes = list(gpd.read_file(path_to_watershed).geometry)
print(f'Basin outline: {len(basin_shapes)} piece(s) read from {path_to_watershed}')

In [ ]:
def values_in_basin(path, shapes):
    """Return (values, is_inside, nodata) for a raster, cropped to the basin's bounding box.

    is_inside is True for cells inside the basin outline."""
    with rasterio.open(path) as r:
        out, _ = mask(r, shapes, crop=True, filled=False)
        values = np.ma.getdata(out[0])
        is_inside = ~np.ma.getmaskarray(out[0])
        return values, is_inside, r.nodata


def valid_basin_values(path, shapes):
    """Return a 1-D array of the raster's values inside the basin, with no-data cells dropped."""
    values, is_inside, nodata = values_in_basin(path, shapes)
    vals = values[is_inside]
    return vals[vals != nodata] if nodata is not None else vals

### Heaviest hourly rain vs Ks (January storm week)

Finds the heaviest rain in any basin radar cell for each hour of Jan 18 to 25 and compares the largest value with the Ks values in the basin, printing the share of basin cells whose Ks is below it.

In [ ]:
# ---- Could the rain outrun the soil? Heaviest hourly rain in the basin vs surface conductivity (Ks) -------------
week_hours = pd.date_range('2017-01-18 01:00', '2017-01-25 00:00', freq='h')   # the hours of the Jan storm week (end-labeled)
rain_max = []                                                                    # heaviest rain in any basin cell, each hour
for t in week_hours:
    path = f'{RADAR_DIR}/{RADAR_BASENAME}{t:%m%d%Y%H}.{RADAR_EXT}'
    values, is_inside, nodata = values_in_basin(path, basin_shapes)
    cells = values[is_inside]
    if nodata is not None:
        cells = cells[cells != nodata]
    rain_max.append(float(cells.max()))
rain_max = np.array(rain_max)
top_rain = rain_max.max()

ks = valid_basin_values(f'{SOLUS_DIR}/solus/Ks_0-5cm.asc', basin_shapes)
print(f'Heaviest hourly rain in any basin radar cell, Jan 18 to 25: {top_rain:.1f} mm/hr '
      f'(hour ending {week_hours[rain_max.argmax()]:%b %d %H:%M})')
print('Ks, 0-5 cm, inside the basin (mm/hr): 5th percentile {:.1f} | median {:.1f} | 95th percentile {:.1f}'.format(
    *np.percentile(ks, [5, 50, 95])))
print(f'Share of basin cells whose Ks is BELOW that rain rate: {(ks < top_rain).mean() * 100:.1f}%')
print('Hours in the week when the heaviest cell got more than 5 mm/hr:', int((rain_max > 5).sum()))

### Screening by storm: what Ks multiplier would each storm need?

For each storm and each Ks multiplier, every Ks cell and hour contributes the rain above (multiplier × Ks) as runoff, and the sum is "the runoff the radar rain could make". The multiplier at which that equals the gauge's volume for the storm (found by interpolation) is the Ks the rain "asks for". If the storms ask for very different multipliers, one Ks multiplier cannot serve them all. The method is rough by design (see the notes at the top of the cell), so compare storms with each other rather than treating one number as correct.

The `STORMS` dictionary here (radar hours and gauge volume) is not the `STORMS` of the rain cell above; each cell defines its own. This cell replaces the earlier single-storm screening cell.

In [ ]:
# ---- Screening by storm: what Ks multiplier would the radar rain need to make each storm's gauge volume? ----------------
# Same idea as ks_screen_cell.py ("rain faster than the soil absorbs it makes runoff"), but for several storms in one go.
# No model run. Read-only: it writes nothing. Run the two "values and helper functions" cells above it first: they set
# RADAR_DIR, RADAR_BASENAME, RADAR_EXT, SOLUS_DIR and basin_shapes (the same names ks_screen_cell.py uses).
#
# For each storm and each Ks multiplier: in every Ks cell and every hour, rain above (multiplier x Ks) cannot soak in.
# Add that up over the basin = "runoff it could make". The multiplier at which that equals the gauge's volume for the
# storm (found by interpolation) is the Ks the radar rain "asks for". If the storms ask for very different multipliers,
# one Ks multiplier cannot serve them all.
#
# Rough by design: it overestimates (soil can absorb more than Ks at the start of a storm), underestimates (bursts shorter
# than an hour are lost in hourly radar) and ignores channel loss and travel. Compare storms with each other, not with
# a single "correct" number.

KS_MULTIPLIERS = [1, 0.7, 0.5, 0.35, 0.25, 0.2, 0.15, 0.1, 0.07, 0.05, 0.03, 0.02]
# label: (first end-labeled radar hour, last end-labeled radar hour, gauge outflow in million m3 for the storm AND its tail)
# Gauge volumes are from the storm_rain_cell.py windows (Jan 18 to Feb 3, Feb 18 to 27, Feb 27 to Mar 12).
STORMS = {'Jan 18-25 flood':  ('2017-01-18 01:00', '2017-01-25 00:00', 8.894),
          'Feb 18-19 storm':  ('2017-02-18 01:00', '2017-02-20 00:00', 1.611),
          'Feb 27-28 storm':  ('2017-02-27 01:00', '2017-03-01 00:00', 4.905)}

# 1. Every Ks cell inside the basin: its value, its area and its position on the map (once for all storms)
with rasterio.open(f'{SOLUS_DIR}/solus/Ks_0-5cm.asc') as r:
    out, ks_transform = mask(r, basin_shapes, crop=True, filled=False)
    ks_nodata = r.nodata
ks_data = np.ma.getdata(out[0])
inside = ~np.ma.getmaskarray(out[0])
if ks_nodata is not None:
    inside &= ks_data != ks_nodata
rows, cols = np.where(inside)
ks = ks_data[rows, cols].astype(float)
x, y = ks_transform * (cols + 0.5, rows + 0.5)         # map position of each cell centre
cell_area = abs(ks_transform.a * ks_transform.e)       # m2 of one Ks cell

results = {}      # storm -> (rain volume, array of runoff volumes, gauge volume)
for storm, (first, last, gauge_vol) in STORMS.items():
    hours = pd.date_range(first, last, freq='h')
    grids = []
    for t in hours:
        with rasterio.open(f'{RADAR_DIR}/{RADAR_BASENAME}{t:%m%d%Y%H}.{RADAR_EXT}') as r:
            grids.append(r.read(1, masked=True).filled(0.0))
            radar_transform = r.transform
    rain = np.array(grids)                              # (hour, row, column), mm in that hour

    # 2. The radar cell that sits over each Ks cell, and the rain it got in every hour
    col_f, row_f = ~radar_transform * (x, y)
    radar_col, radar_row = np.floor(col_f).astype(int), np.floor(row_f).astype(int)
    on_grid = (radar_row >= 0) & (radar_row < rain.shape[1]) & (radar_col >= 0) & (radar_col < rain.shape[2])
    ks_here, rr, rc = ks[on_grid], radar_row[on_grid], radar_col[on_grid]
    rain_at_cell = rain[:, rr, rc]                      # shape (hours, Ks cells)

    rain_vol = rain_at_cell.sum() / 1000 * cell_area / 1e6                      # million m3
    runoff = np.array([np.maximum(rain_at_cell - m * ks_here, 0.0).sum() / 1000 * cell_area / 1e6
                       for m in KS_MULTIPLIERS])
    results[storm] = (rain_vol, runoff, gauge_vol, len(hours), len(ks_here))

# 3. Print: runoff it could make for each multiplier, then the multiplier that matches the gauge
print('SCREENING BY STORM: runoff the radar rain could make (million m3) at each Ks multiplier')
table = pd.DataFrame({s: v[1] for s, v in results.items()}, index=[f'x{m:g}' for m in KS_MULTIPLIERS])
table.index.name = 'Ks multiplier'
print(table.to_string(float_format=lambda v: f'{v:.2f}'))
print()
for storm, (rain_vol, runoff, gauge_vol, n_hours, n_cells) in results.items():
    mults = np.array(KS_MULTIPLIERS, dtype=float)
    order = np.argsort(runoff)                          # runoff grows as the multiplier falls
    if gauge_vol > runoff.max():
        match = f'lower than the smallest multiplier tried (x{mults.min():g})'
    elif gauge_vol < runoff.min():
        match = f'higher than the largest multiplier tried (x{mults.max():g})'
    else:
        match = f'about x{np.exp(np.interp(gauge_vol, runoff[order], np.log(mults)[order])):.2f}'
    print(f'{storm}: {n_hours} radar hours, {n_cells} Ks cells; radar rain {rain_vol:.1f} M m3; '
          f'gauge outflow {gauge_vol:.2f} M m3 ({gauge_vol / rain_vol:.3f} of the rain). Matching Ks multiplier: {match}')
print('\nKs cells are the 100 m SOLUS cells inside the basin, so the rain volumes can differ slightly from the')
print('storm_rain_cell.py values (which use the model mesh areas). The gauge totals include each storm\'s recession tail.')

## Check the mesh: Voronoi polygons that tRIBS wrote

tRIBS gives each mesh node a Voronoi polygon (the area closest to that node). This cell checks those polygons, so it is a check of the mesh itself, but tRIBS writes the polygon files only when it runs, which is why it is here and not in the build notebook.

It reads the polygon file (`Sycamore_voi`) and the node parameter file (`Sycamore.0168_00i`) from a run whose results name is `Sycamore` (a 168-hour run), and prints how many polygons are twisted, drawn clockwise, more than 5% different in area from the area tRIBS used, tiny (under 1 m2), or have a corner more than 500 m from the node, then lists the worst of the flagged nodes. The file names are fixed in the cell; change them to look at another run. The cell finds both files by searching every folder below the working folder, so they only need to exist somewhere under `lab/`. It needs only the Imports cell.

In [ ]:
# --- Voronoi polygons that tRIBS wrote (one polygon per node) ---
voi_path = sorted(glob.glob('**/Sycamore_voi', recursive=True))[0]
polys = {}                      # node ID -> (x, y, corners)
node, corners = None, []
with open(voi_path) as fh:
    for line in fh:
        line = line.strip()
        if line == 'END':
            if node is not None:
                polys[node[0]] = (node[1], node[2], np.array(corners))
            node, corners = None, []
        elif node is None:
            a = line.split(',')
            node = (int(a[0]), float(a[1]), float(a[2]))
        else:
            px, py = line.split(',')
            corners.append((float(px), float(py)))

# --- the area tRIBS itself used for each node (VAr column of the parameter file) ---
int_path = sorted(glob.glob('**/Sycamore.0168_00i', recursive=True))[0]
par = pd.read_csv(int_path, usecols=['ID', 'BndCd', 'VAr', 'FWidth']).set_index('ID')

rows = []
for k, (x, y, c) in polys.items():
    signed = 0.5 * (np.dot(c[:, 0], np.roll(c[:, 1], -1)) - np.dot(c[:, 1], np.roll(c[:, 0], -1)))
    valid = Polygon(c).is_valid
    spread = np.hypot(c[:, 0] - x, c[:, 1] - y).max()          # farthest corner from the node (m)
    rows.append((k, x, y, len(c), signed, valid, spread))
scan = pd.DataFrame(rows, columns=['ID', 'x', 'y', 'corners', 'signed_area', 'valid', 'farthest_corner_m']).set_index('ID')
scan = scan.join(par)
scan['area_gap_pct'] = (100 * (scan['VAr'] - scan['signed_area'].abs()) / scan['VAr'].clip(lower=1)).clip(-999, 999)

n = len(scan)
print(f'{n} nodes | sum of tRIBS areas = {scan.VAr.sum() / 1e6:.1f} km2 | sum of polygon-file areas = {scan.signed_area.abs().sum() / 1e6:.1f} km2')
print(f'twisted / invalid polygons:            {(~scan.valid).sum()}')
print(f'clockwise (negative signed area):      {(scan.signed_area < 0).sum()}')
print(f'tRIBS area and polygon area differ >5%: {(scan.area_gap_pct.abs() > 5).sum()}')
print(f'tRIBS area under 1 m2:                 {(scan.VAr < 1).sum()}')
print(f'corner farther than 500 m from node:   {(scan.farthest_corner_m > 500).sum()}')

bad = scan[(~scan.valid) | (scan.signed_area < 0) | (scan.area_gap_pct.abs() > 5) | (scan.VAr < 1)]
print(f'\nNodes with at least one problem: {len(bad)}  (BndCd 0 = interior, 3 = stream)')
print(bad.BndCd.value_counts().to_dict())
print(bad.sort_values('farthest_corner_m', ascending=False).head(15)[
    ['BndCd', 'x', 'y', 'corners', 'signed_area', 'VAr', 'area_gap_pct', 'farthest_corner_m']].round(1).to_string())